In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [2]:
import json
import os

from openai import OpenAI
from google.colab import userdata

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking
from langchain_community.vectorstores import Chroma  # Vector database
from langchain_openai import OpenAIEmbeddings  # OpenAI embedding function
from langchain_core.documents import Document  # Document abstraction


from langchain_text_splitters import (  # Various splitting strategies
    RecursiveCharacterTextSplitter,  # Best general-purpose splitter
    CharacterTextSplitter,  # Simple split by character count
    MarkdownHeaderTextSplitter,  # Splits based on markdown headers
    HTMLHeaderTextSplitter  # Splits based on HTML tags
)


/tmp/ipykernel_489/929496560.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker  # AI-powered semantic chunking


In [4]:
# =============================================================================
# LOAD DATA: Support Tickets
# =============================================================================
print("\nLoading support tickets...")
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"Loaded {len(tickets)} support tickets")


Loading support tickets...
Loaded 20 support tickets


In [5]:
# Display sample ticket - understand what we're working with
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")

print("\n" + "="*80)
print("PART 1: Chunking Strategies")
print("="*80)


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High

PART 1: Chunking Strategies


In [6]:
# -----------------------------------------------------------------------------
# First, convert our ticket data into LangChain Document objects
# Document = page_content (text) + metadata (structured info for filtering)
# -----------------------------------------------------------------------------

documents = []
for ticket in tickets:
    # Combine all ticket fields into a single text block
    # TIP: Include all relevant context that helps understand the document
    full_text = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
    """.strip()


    # Create Document object with metadata
    # Metadata is CRUCIAL - it enables filtering later!
    # Example: "Find similar tickets, but only in the 'Authentication' category"
    doc = Document(
        page_content=full_text,  # The actual text content
        metadata={
            'ticket_id': ticket['ticket_id'],   # For identifying results
            'category': ticket['category'],      # For category filtering
            'priority': ticket['priority']       # For priority filtering
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nSample document length: {len(documents[0].page_content)} characters")

Created 20 documents

Sample document length: 523 characters


In [8]:
# =============================================================================
# STRATEGY 1: Fixed-Size Chunking with 200 Chucks
# =============================================================================
print("  " + "-"*70)
print("\n--- Strategy 1: Fixed-Size Chunking with 200 Chucks ---")

fixed_splitter1 = CharacterTextSplitter(
    chunk_size=200,      # Maximum characters per chunk
    chunk_overlap=20,    # Characters to repeat between chunks (10% overlap)
    separator="\n"       # Prefer splitting on newlines when possible
)
fixed_chunks1 = fixed_splitter1.split_documents(documents)

print(f"✓ Created {len(fixed_chunks1)} chunks")
print(f"  Chunk size: 200 chars, Overlap: 20 chars")
print(f"  Sample chunk: {fixed_chunks1[0].page_content[:100]}...")
print("  " + "-"*70)

  ----------------------------------------------------------------------

--- Strategy 1: Fixed-Size Chunking with 200 Chucks ---
✓ Created 60 chunks
  Chunk size: 200 chars, Overlap: 20 chars
  Sample chunk: Ticket ID: TICK-001
Title: Users unable to log in after password reset
Category: Authentication
Prio...
  ----------------------------------------------------------------------


In [10]:
# =============================================================================
# STRATEGY 1: Fixed-Size Chunking with 200 Chucks
# =============================================================================
print("  " + "-"*70)
print("\n--- Strategy 1: Fixed-Size Chunking with 500 Chucks ---")

fixed_splitter2 = CharacterTextSplitter(
    chunk_size=500,      # Maximum characters per chunk
    chunk_overlap=50,    # Characters to repeat between chunks (10% overlap)
    separator="\n"       # Prefer splitting on newlines when possible
)
fixed_chunks2 = fixed_splitter1.split_documents(documents)

print(f"✓ Created {len(fixed_chunks2)} chunks")
print(f"  Chunk size: 500 chars, Overlap: 50 chars")
print(f"  Sample chunk: {fixed_chunks2[0].page_content[:100]}...")
print("  " + "-"*70)

  ----------------------------------------------------------------------

--- Strategy 1: Fixed-Size Chunking with 500 Chucks ---
✓ Created 60 chunks
  Chunk size: 500 chars, Overlap: 50 chars
  Sample chunk: Ticket ID: TICK-001
Title: Users unable to log in after password reset
Category: Authentication
Prio...
  ----------------------------------------------------------------------
